In [ ]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D5 — IMF Country Focus — India
# ============================================================

!pip install PyMuPDF
from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D5"

DOCUMENT_NAME = (
    "IMF Country Focus — Booming India at risk of overheating"
)

DOCUMENT_SHORT_NAME = (
    "IMF Country Focus — India Article"
)

SOURCE_FORMAT = "PDF"

EXPECTED_PAGE_COUNT = 2

EXPECTED_REFERENCE_RECORD_COUNT = 44

EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

REFERENCE_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

OUTPUT_DIR = Path(
    "outputs_D5_stage1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Upload original D5 PDF
# ============================================================

print(
    "Upload the original D5 PDF."
)

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:

    raise ValueError(
        "Upload exactly one PDF file."
    )

SOURCE_PATH = pdf_files[0]

print(
    "Uploaded source:",
    SOURCE_PATH.name
)

print(
    "File size:",
    f"{SOURCE_PATH.stat().st_size:,} bytes"
)

In [ ]:
# ============================================================
# 3. File-hashing utility
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print(
    "Source SHA-256:",
    SOURCE_SHA256
)

In [ ]:
# ============================================================
# 4. Open and validate source PDF
# ============================================================

document = fitz.open(
    SOURCE_PATH
)

page_count = len(
    document
)

page_count_valid = (
    page_count
    == EXPECTED_PAGE_COUNT
)

pdf_is_encrypted = bool(
    document.is_encrypted
)

pdf_needs_password = bool(
    document.needs_pass
)


print(
    "Page count:",
    page_count
)

print(
    "Expected page count:",
    EXPECTED_PAGE_COUNT
)

print(
    "Page count valid:",
    page_count_valid
)

print(
    "Encrypted:",
    pdf_is_encrypted
)

print(
    "Needs password:",
    pdf_needs_password
)


if not page_count_valid:

    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {page_count}."
    )

if pdf_needs_password:

    raise ValueError(
        "The source PDF requires a password."
    )

In [ ]:
# ============================================================
# 5. Extract page text and layout blocks
# ============================================================

page_records = []

all_text_parts = []


for page_index, page in enumerate(
    document
):

    page_number = (
        page_index
        + 1
    )

    page_text = page.get_text(
        "text",
        sort=True
    )

    text_blocks = page.get_text(
        "blocks",
        sort=True
    )

    image_list = page.get_images(
        full=True
    )

    page_records.append(
        {
            "Page Number":
                page_number,

            "Text":
                page_text,

            "Text Characters":
                len(
                    page_text
                ),

            "Word Count":
                len(
                    page_text.split()
                ),

            "Text Block Count":
                len(
                    text_blocks
                ),

            "Embedded Image Count":
                len(
                    image_list
                ),

            "Width Points":
                float(
                    page.rect.width
                ),

            "Height Points":
                float(
                    page.rect.height
                )
        }
    )

    all_text_parts.append(
        page_text
    )


full_text = "\n".join(
    all_text_parts
)

page_characterisation_df = pd.DataFrame(
    page_records
)


print(
    "Total text characters:",
    len(
        full_text
    )
)

print(
    "Total words:",
    len(
        full_text.split()
    )
)

display(
    page_characterisation_df
)

In [ ]:
# ============================================================
# 6. Detect document components
# ============================================================

EXPECTED_COMPONENTS = {
    "article_title":
        "Booming India at risk of overheating",

    "country_profile":
        "India at a glance",

    "chart":
        "Taking off",

    "monetary_section":
        "Ease up on the monetary accelerator",

    "fiscal_section":
        "Reduce debt to finance development",

    "capital_markets_section":
        "Develop broader and deeper capital markets",

    "employment_section":
        "Promote job growth and bolster the infrastructure",

    "statistical_table":
        "Inflation risks",

    "author":
        "Charles Kramer",

    "publication_date":
        "April 11, 2007"
}


component_checks = {
    component_name:
        component_text
        in full_text

    for component_name, component_text
    in EXPECTED_COMPONENTS.items()
}


all_expected_components_present = all(
    component_checks.values()
)


print(
    json.dumps(
        component_checks,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "All expected components present:",
    all_expected_components_present
)

if not all_expected_components_present:

    missing_components = [
        component_name
        for component_name, present
        in component_checks.items()
        if not present
    ]

    raise ValueError(
        "Missing expected D5 components: "
        f"{missing_components}"
    )

In [ ]:
# ============================================================
# 7. Quantitative-token diagnostics
# ============================================================

numeric_pattern = re.compile(
    r"(?<!\w)[–−-]?\d+(?:,\d{3})*(?:\.\d+)?"
)

percentage_pattern = re.compile(
    r"\b\d+(?:\.\d+)?\s*percent\b"
    r"|\b\d+(?:\.\d+)?%",
    flags=re.IGNORECASE
)

currency_pattern = re.compile(
    r"\$\s?\d+(?:,\d{3})*(?:\.\d+)?"
)

fiscal_period_pattern = re.compile(
    r"\b(?:19|20)\d{2}/\d{2}\b"
)

calendar_year_pattern = re.compile(
    r"\b(?:19|20)\d{2}\b"
)

basis_point_pattern = re.compile(
    r"\b\d+(?:\.\d+)?\s+basis points?\b",
    flags=re.IGNORECASE
)


numeric_tokens = numeric_pattern.findall(
    full_text
)

percentage_tokens = percentage_pattern.findall(
    full_text
)

currency_tokens = currency_pattern.findall(
    full_text
)

fiscal_periods = sorted(
    set(
        fiscal_period_pattern.findall(
            full_text
        )
    )
)

calendar_years = sorted(
    set(
        calendar_year_pattern.findall(
            full_text
        )
    )
)

basis_point_tokens = basis_point_pattern.findall(
    full_text
)


word_count = len(
    full_text.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)


quantitative_diagnostics = {
    "numeric_token_count":
        len(
            numeric_tokens
        ),

    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),

    "percentage_token_count":
        len(
            percentage_tokens
        ),

    "currency_token_count":
        len(
            currency_tokens
        ),

    "basis_point_token_count":
        len(
            basis_point_tokens
        ),

    "detected_fiscal_periods":
        fiscal_periods,

    "detected_calendar_years":
        calendar_years
}


print(
    json.dumps(
        quantitative_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 8. Create document metadata
# ============================================================

pdf_metadata = document.metadata or {}


DOCUMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "short_name":
        DOCUMENT_SHORT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "file_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        int(
            SOURCE_PATH.stat().st_size
        ),

    "page_count":
        page_count,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        bool(
            page_count_valid
        ),

    "pdf_title_metadata":
        pdf_metadata.get(
            "title"
        ),

    "pdf_author_metadata":
        pdf_metadata.get(
            "author"
        ),

    "pdf_subject_metadata":
        pdf_metadata.get(
            "subject"
        ),

    "pdf_creator_metadata":
        pdf_metadata.get(
            "creator"
        ),

    "pdf_producer_metadata":
        pdf_metadata.get(
            "producer"
        ),

    "encrypted":
        pdf_is_encrypted,

    "password_required":
        pdf_needs_password,

    "text_extractable":
        bool(
            full_text.strip()
        ),

    "total_text_characters":
        len(
            full_text
        ),

    "total_words":
        len(
            full_text.split()
        ),

    "language":
        "English",

    "publication_date":
        "April 11, 2007",

    "publisher":
        "International Monetary Fund",

    "article_author":
        "Charles Kramer",

    "article_department":
        "IMF Asia and Pacific Department"
}


print(
    json.dumps(
        DOCUMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 9. Create document characterisation
# ============================================================

DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "document_type":
        "Two-page magazine-style economic policy article",

    "source_format":
        SOURCE_FORMAT,

    "page_count":
        page_count,

    "text_layer_present":
        bool(
            full_text.strip()
        ),

    "ocr_required":
        False,

    "reading_order_complexity":
        "High",

    "multi_column_layout":
        True,

    "contains_narrative_text":
        True,

    "contains_section_headings":
        True,

    "contains_bulleted_policy_measures":
        True,

    "contains_country_profile_box":
        component_checks[
            "country_profile"
        ],

    "contains_chart":
        component_checks[
            "chart"
        ],

    "contains_statistical_table":
        component_checks[
            "statistical_table"
        ],

    "contains_photograph":
        True,

    "contains_promotional_sidebar":
        True,

    "contains_footnote":
        True,

    "contains_negative_values":
        True,

    "contains_rounded_narrative_values":
        True,

    "contains_qualifying_language": {
        "about":
            "about" in full_text.lower(),

        "more_than":
            "more than" in full_text.lower(),

        "nearly":
            "nearly" in full_text.lower(),

        "just_over":
            "just over" in full_text.lower(),

        "around":
            "around" in full_text.lower()
    },

    "contains_mixed_units":
        True,

    "contains_mixed_reference_periods":
        True,

    "main_information_regions": [
        "Narrative article body",
        "Four principal policy-measure bullets",
        "India at a glance profile box",
        "Taking off chart caption",
        "Inflation risks statistical table"
    ],

    "excluded_non-task_regions": [
        "Chart-line values requiring visual estimation",
        "Photograph and photograph caption",
        "IMF Website for Legislators promotional box",
        "Page numbers",
        "Copyright footer",
        "Publisher branding"
    ],

    "quantitative_diagnostics":
        quantitative_diagnostics,

    "expected_components_present":
        component_checks,

    "all_expected_components_present":
        bool(
            all_expected_components_present
        )
}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 10. Define fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed set of policy and quantitative records represented
in the IMF Country Focus article “Booming India at risk of overheating.”

Scope:

Include:

1. The four principal policy measures introduced by the sentence
   “A combination of these four main policy measures is critical”.
2. Every item in the “India at a glance” country-profile box.
3. Every explicitly stated quantitative observation in the narrative
   article that belongs to the predefined reference scope.
4. The explicit quantitative statement in the “Taking off” chart
   caption.
5. Every numeric observation in the “Inflation risks” statistical
   table.

Exclude:

- values inferred or estimated from the plotted chart lines;
- page numbers and publication metadata;
- the photograph and photograph caption;
- promotional content;
- qualitative statements without a requested policy measure or
  quantitative value;
- detailed policy sub-actions that elaborate the four principal policy
  measures;
- values appearing only in source citations or copyright text.

For every included record, return:

- Category
- Indicator or Policy Area
- Value
- Unit
- Qualifier
- Reference Period
- Description
- Source Location

Rules:

- Preserve qualifiers such as “about”, “more than”, “nearly”,
  “just over” and “around” in the Qualifier field.
- Use null when no qualifier or reference period is explicitly
  associated with a record.
- Do not silently correct or reinterpret source units.
- In particular, preserve the unit printed for Gross reserves in the
  statistical table.
- Do not calculate, infer, reconstruct or estimate unsupported values.
- Return exactly 44 records.
- Return the result as valid JSON using the exact field names defined
  in the extraction schema.
- Do not include explanations before or after the JSON.
"""

print(
    EXTRACTION_TASK
)

In [ ]:
# ============================================================
# 11. Define reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        (
            "Principal policy measure, country-profile item, "
            "narrative quantitative observation or statistical-table "
            "observation"
        ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category":
            (
                "One of Main policy measure, Country profile, "
                "Narrative quantitative observation or Statistical "
                "table observation"
            ),

        "Indicator or Policy Area":
            (
                "The policy area, profile item or quantitative "
                "indicator represented by the record"
            ),

        "Value":
            (
                "The numeric or textual value explicitly represented "
                "in the source"
            ),

        "Unit":
            (
                "The source-grounded measurement unit or text for "
                "non-numeric policy/profile values"
            ),

        "Qualifier":
            (
                "An explicit approximation or inequality qualifier, "
                "such as about, more than, nearly, just over or around"
            ),

        "Reference Period":
            (
                "The explicitly associated year, fiscal period, "
                "duration or relative period"
            ),

        "Description":
            (
                "A concise source-grounded description of what the "
                "record represents"
            ),

        "Source Location":
            (
                "The page and named article region containing the "
                "record"
            )
    },

    "null_policy": {
        "Qualifier":
            "null when the source provides no qualifier",

        "Reference Period":
            (
                "null when no explicit period or duration is "
                "associated with the record"
            )
    },

    "construction_method":
        (
            "Manual document-grounded transcription followed by "
            "programmatic integrity checks"
        ),

    "branch_reuse":
        (
            "The same fixed reference dataset is reused for "
            "Branches A, B and C"
        )
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 12. Define extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        (
            "Principal policy measure, country-profile item, "
            "narrative quantitative observation or "
            "statistical-table observation"
        ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": [
                "string",
                "null"
            ]
        },

        "Indicator or Policy Area": {
            "type": [
                "string",
                "null"
            ]
        },

        "Value": {
            "type": [
                "string",
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": [
                "string",
                "null"
            ]
        },

        "Qualifier": {
            "type": [
                "string",
                "null"
            ]
        },

        "Reference Period": {
            "type": [
                "string",
                "null"
            ]
        },

        "Description": {
            "type": [
                "string",
                "null"
            ]
        },

        "Source Location": {
            "type": [
                "string",
                "null"
            ]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Indicator or Policy Area":
                    "string or null",

                "Value":
                    "string, number or null",

                "Unit":
                    "string or null",

                "Qualifier":
                    "string or null",

                "Reference Period":
                    "string or null",

                "Description":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}


print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 13. Construct fixed reference records
# ============================================================

reference_records = [
    # ========================================================
    # A. Four principal policy measures
    # ========================================================

    {
        "Category":
            "Main policy measure",

        "Indicator or Policy Area":
            "Price and financial stability",

        "Value":
            (
                "Managing price and financial stability by limiting "
                "the near-term risk of overheating in demand and by "
                "further strengthening financial regulation."
            ),

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            (
                "Principal policy measure concerning overheating, "
                "inflation and financial regulation"
            ),

        "Source Location":
            "Page 1 — Four main policy measures"
    },

    {
        "Category":
            "Main policy measure",

        "Indicator or Policy Area":
            "Fiscal sustainability while financing development",

        "Value":
            (
                "Achieving fiscal sustainability while financing "
                "development by reducing high debt and making "
                "budgetary room to fund social and infrastructure "
                "spending."
            ),

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            (
                "Principal policy measure concerning debt reduction "
                "and priority spending"
            ),

        "Source Location":
            "Page 1 — Four main policy measures"
    },

    {
        "Category":
            "Main policy measure",

        "Indicator or Policy Area":
            "Financial sector development",

        "Value":
            (
                "Broadening and deepening the financial sector to "
                "expand the channels for saving, investment, and "
                "risk management."
            ),

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            (
                "Principal policy measure concerning financial "
                "intermediation and risk management"
            ),

        "Source Location":
            "Page 1 — Four main policy measures"
    },

    {
        "Category":
            "Main policy measure",

        "Indicator or Policy Area":
            "Job-intensive inclusive growth",

        "Value":
            (
                "Promoting more job-intensive, inclusive growth "
                "through further structural reforms to create an "
                "environment in which growth more fully benefits "
                "the least advantaged segments of the population."
            ),

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            (
                "Principal policy measure concerning employment, "
                "inclusion and structural reform"
            ),

        "Source Location":
            "Page 1 — Four main policy measures"
    },

    # ========================================================
    # B. India at a glance
    # ========================================================

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "Capital",

        "Value":
            "New Delhi",

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            "Capital of India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "Area",

        "Value":
            2973190,

        "Unit":
            "sq. km.",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            "Area of India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "Population",

        "Value":
            1.11,

        "Unit":
            "billion people",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "Population of India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "Life expectancy",

        "Value":
            64.71,

        "Unit":
            "years",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            "Life expectancy in India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "GDP per capita",

        "Value":
            716,

        "Unit":
            "USD",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "GDP per capita in India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    {
        "Category":
            "Country profile",

        "Indicator or Policy Area":
            "Main exports",

        "Value":
            (
                "Software and information technology services, "
                "textiles, jewelry, agriculture, engineering goods, "
                "and chemicals"
            ),

        "Unit":
            "text",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            "Main exports listed for India",

        "Source Location":
            "Page 1 — India at a glance"
    },

    # ========================================================
    # C. Narrative and chart-caption quantitative observations
    # ========================================================

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Average GDP growth",

        "Value":
            8.5,

        "Unit":
            "percent",

        "Qualifier":
            None,

        "Reference Period":
            "four years running",

        "Description":
            "Average GDP growth over four consecutive years",

        "Source Location":
            "Page 1 — Opening narrative"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Real per capita income doubling interval",

        "Value":
            13,

        "Unit":
            "years",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            (
                "Interval in which real per capita income would "
                "double according to the IMF trend-growth estimate"
            ),

        "Source Location":
            "Page 1 — Opening narrative"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Working-age population increase",

        "Value":
            140,

        "Unit":
            "million people",

        "Qualifier":
            None,

        "Reference Period":
            "next 10 years",

        "Description":
            (
                "Expected increase in India's working-age population"
            ),

        "Source Location":
            "Page 1 — Opening narrative"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "GDP growth since 2002",

        "Value":
            8,

        "Unit":
            "percent",

        "Qualifier":
            "around",

        "Reference Period":
            "since 2002",

        "Description":
            (
                "GDP growth level stated in the Taking off chart "
                "caption"
            ),

        "Source Location":
            "Page 1 — Taking off chart caption"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Recent wholesale price inflation",

        "Value":
            6,

        "Unit":
            "percent",

        "Qualifier":
            "more than",

        "Reference Period":
            "recently",

        "Description":
            (
                "Annual rise in the wholesale price index"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Wholesale price inflation one year earlier",

        "Value":
            4,

        "Unit":
            "percent",

        "Qualifier":
            "about",

        "Reference Period":
            "a year ago",

        "Description":
            (
                "Wholesale price inflation approximately one year "
                "before the recent observation"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Main lending rate hike",

        "Value":
            175,

        "Unit":
            "basis points",

        "Qualifier":
            None,

        "Reference Period":
            "after the latest tightening",

        "Description":
            (
                "Updated cumulative hike in the main lending rate"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Previous main lending rate hike",

        "Value":
            150,

        "Unit":
            "basis points",

        "Qualifier":
            None,

        "Reference Period":
            "before the latest tightening",

        "Description":
            (
                "Previously stated cumulative hike in the main "
                "lending rate"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Cash reserve ratio increase",

        "Value":
            150,

        "Unit":
            "basis points",

        "Qualifier":
            None,

        "Reference Period":
            "after the latest tightening",

        "Description":
            (
                "Updated cumulative increase in the cash reserve "
                "ratio"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Previous cash reserve ratio increase",

        "Value":
            100,

        "Unit":
            "basis points",

        "Qualifier":
            None,

        "Reference Period":
            "before the latest tightening",

        "Description":
            (
                "Previously stated cumulative increase in the cash "
                "reserve ratio"
            ),

        "Source Location":
            "Page 1 — Ease up on the monetary accelerator"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "General government deficit",

        "Value":
            6,

        "Unit":
            "percent of GDP",

        "Qualifier":
            "about",

        "Reference Period":
            "2006/07",

        "Description":
            (
                "General government deficit projected for the "
                "2006/07 fiscal year"
            ),

        "Source Location":
            "Page 1 — Reduce debt to finance development"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Previous high general government deficit",

        "Value":
            10,

        "Unit":
            "percent of GDP",

        "Qualifier":
            "just over",

        "Reference Period":
            "2001/02",

        "Description":
            (
                "Earlier high in the general government deficit"
            ),

        "Source Location":
            "Page 1 — Reduce debt to finance development"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Public debt",

        "Value":
            80,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            None,

        "Description":
            "Public debt level described as remaining high",

        "Source Location":
            "Page 2 — Reduce debt to finance development"
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Agricultural labour-force share",

        "Value":
            60,

        "Unit":
            "percent of labour force",

        "Qualifier":
            "nearly",

        "Reference Period":
            None,

        "Description":
            (
                "Share of the labour force continuing to work in "
                "agriculture"
            ),

        "Source Location":
            (
                "Page 2 — Promote job growth and bolster the "
                "infrastructure"
            )
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Population living on less than USD 2 a day",

        "Value":
            800,

        "Unit":
            "million people",

        "Qualifier":
            "more than",

        "Reference Period":
            None,

        "Description":
            (
                "Number of Indians living on less than USD 2 a day"
            ),

        "Source Location":
            (
                "Page 2 — Promote job growth and bolster the "
                "infrastructure"
            )
    },

    {
        "Category":
            "Narrative quantitative observation",

        "Indicator or Policy Area":
            "Infrastructure needs",

        "Value":
            300,

        "Unit":
            "USD billion",

        "Qualifier":
            "more than",

        "Reference Period":
            "medium term",

        "Description":
            "Estimated infrastructure financing needs",

        "Source Location":
            (
                "Page 2 — Promote job growth and bolster the "
                "infrastructure"
            )
    },

    # ========================================================
    # D. Inflation risks statistical table
    # ========================================================

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Real GDP",

        "Value":
            7.5,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            "Real GDP growth",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Real GDP",

        "Value":
            9.0,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "Real GDP growth, provisional",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Real GDP",

        "Value":
            8.9,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            "Real GDP growth, estimate",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Wholesale prices",

        "Value":
            6.5,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            "Wholesale-price change",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Wholesale prices",

        "Value":
            4.4,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "Wholesale-price change, provisional",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Wholesale prices",

        "Value":
            6.4,

        "Unit":
            "percent change",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            (
                "Wholesale-price change, estimate; table footnote "
                "states the observation is as of the week ended "
                "March 24, 2007"
            ),

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "General government debt",

        "Value":
            85.7,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            "General government debt",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "General government debt",

        "Value":
            81.9,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "General government debt, provisional",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "General government debt",

        "Value":
            79.3,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            "General government debt, estimate",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Current account balance",

        "Value":
            -2.5,

        "Unit":
            "billion dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            "Current account balance",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Current account balance",

        "Value":
            -9.1,

        "Unit":
            "billion dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "Current account balance, provisional",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Current account balance",

        "Value":
            -22.7,

        "Unit":
            "billion dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            "Current account balance, estimate",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "External debt",

        "Value":
            17.7,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            "External debt",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "External debt",

        "Value":
            15.7,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            "External debt, provisional",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "External debt",

        "Value":
            18.1,

        "Unit":
            "percent of GDP",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            "External debt, estimate",

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Gross reserves",

        "Value":
            141.5,

        "Unit":
            "million dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2004/05",

        "Description":
            (
                "Gross reserves using the unit printed in the table"
            ),

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Gross reserves",

        "Value":
            151.6,

        "Unit":
            "million dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2005/06",

        "Description":
            (
                "Gross reserves, provisional, using the unit printed "
                "in the table"
            ),

        "Source Location":
            "Page 2 — Inflation risks table"
    },

    {
        "Category":
            "Statistical table observation",

        "Indicator or Policy Area":
            "Gross reserves",

        "Value":
            198.6,

        "Unit":
            "million dollars",

        "Qualifier":
            None,

        "Reference Period":
            "2006/07",

        "Description":
            (
                "Gross reserves, estimate, using the unit printed "
                "in the table"
            ),

        "Source Location":
            "Page 2 — Inflation risks table"
    }
]


reference_values_df = pd.DataFrame(
    reference_records,
    columns=REFERENCE_FIELDS
)


print(
    "Reference records constructed:",
    len(
        reference_values_df
    )
)

display(
    reference_values_df
)

In [ ]:
# ============================================================
# 14. Validate reference schema
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == REFERENCE_FIELDS
)

record_count_valid = (
    len(
        reference_values_df
    )
    == EXPECTED_REFERENCE_RECORD_COUNT
)


print(
    "Reference schema valid:",
    reference_schema_valid
)

print(
    "Observed records:",
    len(
        reference_values_df
    )
)

print(
    "Expected records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)

print(
    "Record count valid:",
    record_count_valid
)


if not reference_schema_valid:

    raise AssertionError(
        "The D5 reference schema is invalid."
    )

if not record_count_valid:

    raise AssertionError(
        "The D5 reference record count is invalid."
    )

In [ ]:
# ============================================================
# 15. Validate category counts
# ============================================================

observed_category_counts = (
    reference_values_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)


category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


print(
    "Expected category counts:"
)

print(
    json.dumps(
        EXPECTED_CATEGORY_COUNTS,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "\nCategory counts valid:",
    category_counts_valid
)


if not category_counts_valid:

    raise AssertionError(
        "The D5 category counts are invalid."
    )

In [ ]:
# ============================================================
# 16. Validate required fields and duplicate keys
# ============================================================

REQUIRED_NON_NULL_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Description",
    "Source Location"
]


missing_required_values = {
    field: int(
        reference_values_df[
            field
        ].isna().sum()
    )

    for field in REQUIRED_NON_NULL_FIELDS
}


duplicate_key_mask = (
    reference_values_df.duplicated(
        subset=[
            "Category",
            "Indicator or Policy Area",
            "Reference Period",
            "Source Location"
        ],
        keep=False
    )
)


duplicate_reference_keys_df = (
    reference_values_df.loc[
        duplicate_key_mask
    ].copy()
)


missing_required_value_count = sum(
    missing_required_values.values()
)

duplicate_reference_key_count = len(
    duplicate_reference_keys_df
)


print(
    "Missing required values:"
)

print(
    json.dumps(
        missing_required_values,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Duplicate reference-key records:",
    duplicate_reference_key_count
)


if not duplicate_reference_keys_df.empty:

    display(
        duplicate_reference_keys_df
    )

if missing_required_value_count > 0:

    raise AssertionError(
        "D5 reference records contain missing required values."
    )


if duplicate_reference_key_count > 0:

    raise AssertionError(
        "Duplicate D5 reference keys were detected."
    )

In [ ]:
# ============================================================
# 17. Validate source locations
# ============================================================

ALLOWED_SOURCE_LOCATIONS = {
    "Page 1 — Four main policy measures",
    "Page 1 — India at a glance",
    "Page 1 — Opening narrative",
    "Page 1 — Taking off chart caption",
    "Page 1 — Ease up on the monetary accelerator",
    "Page 1 — Reduce debt to finance development",
    "Page 2 — Reduce debt to finance development",
    "Page 2 — Promote job growth and bolster the infrastructure",
    "Page 2 — Inflation risks table"
}


unexpected_source_locations = sorted(
    set(
        reference_values_df[
            "Source Location"
        ]
    )
    - ALLOWED_SOURCE_LOCATIONS
)


source_locations_valid = (
    len(
        unexpected_source_locations
    )
    == 0
)


print(
    "Unexpected source locations:",
    unexpected_source_locations
)

print(
    "Source locations valid:",
    source_locations_valid
)

if not source_locations_valid:

    raise AssertionError(
        "Unexpected D5 source locations were detected."
    )

In [ ]:
# ============================================================
# 18. Validate Inflation risks table reference structure
# ============================================================

table_df = reference_values_df.loc[
    reference_values_df[
        "Category"
    ]
    == "Statistical table observation"
].copy()


expected_table_indicators = {
    "Real GDP",
    "Wholesale prices",
    "General government debt",
    "Current account balance",
    "External debt",
    "Gross reserves"
}

expected_table_periods = {
    "2004/05",
    "2005/06",
    "2006/07"
}


observed_table_indicators = set(
    table_df[
        "Indicator or Policy Area"
    ]
)

observed_table_periods = set(
    table_df[
        "Reference Period"
    ]
)


table_indicator_counts = (
    table_df[
        "Indicator or Policy Area"
    ]
    .value_counts()
    .to_dict()
)


every_table_indicator_has_three_periods = all(
    count == 3
    for count in table_indicator_counts.values()
)


table_structure_valid = all(
    [
        len(
            table_df
        ) == 18,

        observed_table_indicators
        == expected_table_indicators,

        observed_table_periods
        == expected_table_periods,

        every_table_indicator_has_three_periods
    ]
)


print(
    "Table records:",
    len(
        table_df
    )
)

print(
    "Table indicators:",
    sorted(
        observed_table_indicators
    )
)

print(
    "Table periods:",
    sorted(
        observed_table_periods
    )
)

print(
    "Each table indicator has three periods:",
    every_table_indicator_has_three_periods
)

print(
    "Table structure valid:",
    table_structure_valid
)

if not table_structure_valid:

    raise AssertionError(
        "The D5 Inflation risks table structure is invalid."
    )

In [ ]:
# ============================================================
# 19. Create reference summary
# ============================================================

REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(
                reference_values_df
            )
        ),

    "record_count_valid":
        bool(
            record_count_valid
        ),

    "reference_schema_valid":
        bool(
            reference_schema_valid
        ),

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        bool(
            category_counts_valid
        ),

    "missing_values_by_field": {
        column: int(
            reference_values_df[
                column
            ].isna().sum()
        )

        for column
        in reference_values_df.columns
    },

    "missing_required_value_count":
        int(
            missing_required_value_count
        ),

    "duplicate_reference_key_count":
        int(
            duplicate_reference_key_count
        ),

    "source_locations_valid":
        bool(
            source_locations_valid
        ),

    "table_structure_valid":
        bool(
            table_structure_valid
        ),

    "categories":
        reference_values_df[
            "Category"
        ].drop_duplicates().tolist(),

    "units":
        sorted(
            reference_values_df[
                "Unit"
            ].dropna().unique().tolist()
        ),

    "qualifiers":
        sorted(
            reference_values_df[
                "Qualifier"
            ].dropna().unique().tolist()
        ),

    "construction_method":
        (
            "Manual document-grounded transcription followed by "
            "programmatic verification"
        ),

    "manual_calculation_applied":
        False,

    "external_knowledge_used":
        False,

    "chart_values_visually_estimated":
        False,

    "reference_dataset_fixed_across_branches":
        True
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 20. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "High",
        "Evidence Source":
            "PDF text extraction + manual layout inspection",
        "Justification":
            "The two-page article uses a magazine-style multi-column "
            "layout with several independent visual regions, creating "
            "a substantial risk that native text extraction interleaves "
            "content from different parts of the page."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Medium",
        "Evidence Source":
            "Manual table inspection + extracted text inspection",
        "Justification":
            "The Inflation risks table is conceptually regular, but "
            "correct extraction requires preserving six indicators "
            "across three fiscal periods together with grouped units "
            "and provisional or estimated period labels."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Medium",
        "Evidence Source":
            "Manual document inspection",
        "Justification":
            "Narrative headings are clear, but the page also contains "
            "policy bullets, a country-profile box, chart content, "
            "a statistical table, and other visually independent "
            "regions outside a single linear hierarchy."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "The born-digital PDF is visually clear and relevant text "
            "and numerical values are readable."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "No relevant scanning noise, blur, or degradation affects "
            "the document."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated PDF text extraction",
        "Justification":
            "The document contains an embedded machine-readable text "
            "layer and OCR is not required."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual content inspection",
        "Justification":
            "Macroeconomic, fiscal, monetary, demographic, and policy "
            "terminology is used consistently throughout the article."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The extraction schema must represent several semantic "
            "record types, including policy measures, country-profile "
            "facts, narrative quantitative observations, and "
            "statistical-table observations."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "Medium",
        "Evidence Source":
            "Automated quantitative profiling + manual inspection",
        "Justification":
            "The document contains many economic and demographic "
            "values, but quantitative information is embedded within "
            "a predominantly narrative two-page article rather than "
            "forming the dominant document representation."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All information required for the predefined 44-record "
            "extraction scope is explicitly present in the document."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual source and reference inspection",
        "Justification":
            "The document is internally coherent, but related concepts "
            "may appear with different precision or qualifiers across "
            "narrative and tabular regions, requiring careful "
            "context-preserving comparison."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "High",
        "Evidence Source":
            "Document profiling + manual layout inspection",
        "Justification":
            "The article combines narrative prose, policy bullets, "
            "a country-profile box, chart content, a statistical "
            "table, a photograph, and a promotional sidebar."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "High",
        "Evidence Source":
            "Reference and source inspection",
        "Justification":
            "The predefined extraction scope contains percentages, "
            "percent of GDP, percent of labour force, basis points, "
            "years, million and billion people, USD, USD billion, "
            "square kilometres, text values, and approximation "
            "qualifiers such as about, more than, nearly, just over, "
            "and around."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

In [ ]:
# ============================================================
# 21. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: "
        f"{invalid_scores}"
    )


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: "
        f"{missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: "
        f"{unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

In [ ]:
# ============================================================
# 22. Derive dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

In [ ]:
# ============================================================
# 23. Build structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [ ]:
# ============================================================
# 24. Determine overall reference integrity
# ============================================================

REFERENCE_INTEGRITY_PASSED = all(
    [
        page_count_valid,
        bool(
            full_text.strip()
        ),
        all_expected_components_present,
        reference_schema_valid,
        record_count_valid,
        category_counts_valid,
        missing_required_value_count == 0,
        duplicate_reference_key_count == 0,
        source_locations_valid,
        table_structure_valid
    ]
)


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "page_count_valid":
        bool(
            page_count_valid
        ),

    "text_extractable":
        bool(
            full_text.strip()
        ),

    "all_expected_components_present":
        bool(
            all_expected_components_present
        ),

    "reference_schema_valid":
        bool(
            reference_schema_valid
        ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_record_count":
        int(
            len(
                reference_values_df
            )
        ),

    "record_count_valid":
        bool(
            record_count_valid
        ),

    "category_counts_valid":
        bool(
            category_counts_valid
        ),

    "missing_required_value_count":
        int(
            missing_required_value_count
        ),

    "duplicate_reference_key_count":
        int(
            duplicate_reference_key_count
        ),

    "source_locations_valid":
        bool(
            source_locations_valid
        ),

    "table_structure_valid":
        bool(
            table_structure_valid
        ),

    "reference_integrity_passed":
        bool(
            REFERENCE_INTEGRITY_PASSED
        ),

    "construction_method":
        (
            "Manual document-grounded transcription followed by "
            "programmatic integrity validation"
        ),

    "reference_values_manually_verified":
        True,

    "external_knowledge_used":
        False,

    "unsupported_chart_estimation_used":
        False,

    "reference_dataset_fixed_for_branches_A_B_C":
        True
}


print(
    json.dumps(
        REFERENCE_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


if not REFERENCE_INTEGRITY_PASSED:

    raise AssertionError(
        "D5 reference-value integrity failed."
    )

In [ ]:
# ============================================================
# 25. Export extracted page text for audit
# ============================================================

page_text_audit_rows = [
    {
        "Page Number":
            record[
                "Page Number"
            ],

        "Text":
            record[
                "Text"
            ]
    }

    for record in page_records
]


page_text_audit_df = pd.DataFrame(
    page_text_audit_rows
)


PAGE_TEXT_AUDIT_PATH = (
    OUTPUT_DIR
    / "D5_page_text_audit.csv"
)


page_text_audit_df.to_csv(
    PAGE_TEXT_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    PAGE_TEXT_AUDIT_PATH
)

In [ ]:
# ============================================================
# 26. Define output paths
# ============================================================

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR /
    "D5_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR /
    "D5_reference_values.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D5_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D5_dimension_assessment.csv"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D5_extraction_schema.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D5_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR /
    "D5_reference_summary.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR /
    "D5_reference_integrity.json"
)

DOCUMENT_METADATA_PATH = (
    OUTPUT_DIR /
    "D5_document_metadata.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR /
    "D5_document_characterisation.json"
)

PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR /
    "D5_page_characterisation.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR /
    "D5_quality_evidence.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR /
    "D5_extraction_task.txt"
)

NOTEBOOK_SUMMARY_PATH = (
    OUTPUT_DIR /
    "D5_stage1_summary.json"
)

In [ ]:
# ============================================================
# 27. Export Stage 1 outputs
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

reference_values_df.to_json(
    REFERENCE_VALUES_JSON_PATH,
    orient="records",
    indent=2,
    force_ascii=False
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

page_characterisation_df.drop(
    columns=[
        "Text"
    ]
).to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)


EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8"
)


json_outputs = [
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),
    (
        DOCUMENT_METADATA_PATH,
        DOCUMENT_METADATA
    ),
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]


for output_path, content in json_outputs:

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            content,
            file,
            indent=2,
            ensure_ascii=False
        )


print(
    "D5 Stage 1 outputs exported."
)

In [ ]:
# ============================================================
# 28. Stage 1 summary
# ============================================================

STAGE1_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "page_count":
        page_count,

    "text_extractable":
        bool(
            full_text.strip()
        ),

    "ocr_required":
        False,

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(
                reference_values_df
            )
        ),

    "reference_integrity_passed":
        bool(
            REFERENCE_INTEGRITY_PASSED
        ),

    "category_counts":
        observed_category_counts,

    "indicator_assessment":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_assessment":
        dimension_assessment_df.to_dict(
            orient="records"
        ),

    "reference_construction":
        (
            "Manual document-grounded transcription followed by "
            "programmatic verification"
        ),

    "reference_reused_across_branches":
        True,

    "outputs_created": [
        REFERENCE_VALUES_PATH.name,
        REFERENCE_VALUES_JSON_PATH.name,
        INDICATOR_ASSESSMENT_PATH.name,
        DIMENSION_ASSESSMENT_PATH.name,
        EXTRACTION_SCHEMA_PATH.name,
        REFERENCE_SCHEMA_PATH.name,
        REFERENCE_SUMMARY_PATH.name,
        REFERENCE_INTEGRITY_PATH.name,
        DOCUMENT_METADATA_PATH.name,
        DOCUMENT_CHARACTERISATION_PATH.name,
        PAGE_CHARACTERISATION_PATH.name,
        PAGE_TEXT_AUDIT_PATH.name,
        QUALITY_EVIDENCE_PATH.name,
        EXTRACTION_TASK_PATH.name,
        NOTEBOOK_SUMMARY_PATH.name
    ],

}


with NOTEBOOK_SUMMARY_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        STAGE1_SUMMARY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        STAGE1_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 29. Final Stage 1 checks
# ============================================================

print(
    "Page count valid:",
    page_count_valid
)

print(
    "Text extractable:",
    bool(
        full_text.strip()
    )
)

print(
    "Expected document components present:",
    all_expected_components_present
)

print(
    "Reference schema valid:",
    reference_schema_valid
)

print(
    "Reference record count:",
    len(
        reference_values_df
    )
)

print(
    "Reference count valid:",
    record_count_valid
)

print(
    "Category counts valid:",
    category_counts_valid
)

print(
    "Missing required values:",
    missing_required_value_count
)

print(
    "Duplicate reference keys:",
    duplicate_reference_key_count
)

print(
    "Table structure valid:",
    table_structure_valid
)

print(
    "Reference integrity passed:",
    REFERENCE_INTEGRITY_PASSED
)


if not REFERENCE_INTEGRITY_PASSED:

    raise AssertionError(
        "D5 Stage 1 integrity checks failed."
    )


print(
    "\nD5 Stage 1 completed successfully."
)

print(
    "Next step: D5 Branch A — direct PDF ingestion."
)

In [ ]:
# ============================================================
# 30. List generated outputs
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    PAGE_CHARACTERISATION_PATH,
    PAGE_TEXT_AUDIT_PATH,
    QUALITY_EVIDENCE_PATH,
    EXTRACTION_TASK_PATH,
    NOTEBOOK_SUMMARY_PATH
]


print(
    "Generated D5 Stage 1 files:\n"
)


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

In [ ]:
# ============================================================
# 31. Download generated outputs
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            str(
                output_path
            )
        )